# Exercise 2: Calling LLM via API

**Objective**: Learn how to interact with an LLM using an API endpoint and process responses.

**Learning Outcomes**:
- Understand API-based LLM interaction
- Learn prompt structuring and response parsing
- Handle API errors gracefully
- Control response generation with parameters

## Block 1: Import Libraries and Setup

In [1]:
import requests
import json
import os
from dotenv import load_dotenv
import pandas as pd

# Load environment variables
load_dotenv()

print("✓ All libraries imported successfully")
print("\nLibraries:")
print("  - requests: For HTTP API calls")
print("  - json: For JSON parsing")
print("  - dotenv: For environment variables")

✓ All libraries imported successfully

Libraries:
  - requests: For HTTP API calls
  - json: For JSON parsing
  - dotenv: For environment variables


## Block 2: Configure API Credentials

In [2]:
# Initialize API configuration
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
API_URL = "https://api.anthropic.com/v1/messages"
MODEL = "claude-3-5-haiku-20241022"

print("\n[Setup] API Configuration:")
print(f"  API URL: {API_URL}")
print(f"  Model: {MODEL}")

if ANTHROPIC_API_KEY:
    print(f"  ✓ API Key loaded from .env")
    print(f"  ✓ Key length: {len(ANTHROPIC_API_KEY)} characters")
else:
    print("  ❌ ERROR: ANTHROPIC_API_KEY not found in .env file")


[Setup] API Configuration:
  API URL: https://api.anthropic.com/v1/messages
  Model: claude-3-5-haiku-20241022
  ✓ API Key loaded from .env
  ✓ Key length: 25 characters


## Block 3: Define API Call Function

In [3]:
def call_llm_api(user_query, temperature=0.7, max_tokens=1024):
    """
    Send a query to Claude Haiku via API and get response
    
    Args:
        user_query: The user's input query
        temperature: Controls randomness (0.0-1.0)
        max_tokens: Maximum tokens in response
    
    Returns:
        dict: Full response from API
    """
    headers = {
        "x-api-key": ANTHROPIC_API_KEY,
        "anthropic-version": "2023-06-01",
        "content-type": "application/json"
    }

    payload = {
        "model": MODEL,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [
            {
                "role": "user",
                "content": user_query
            }
        ]
    }

    try:
        print(f"  Sending request to API...")
        response = requests.post(API_URL, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        return response.json()

    except requests.exceptions.Timeout:
        return {"error": "Request timeout - API took too long to respond"}
    except requests.exceptions.HTTPError as e:
        return {"error": f"HTTP Error {e.response.status_code}: {e.response.text}"}
    except requests.exceptions.RequestException as e:
        return {"error": f"Request failed: {str(e)}"}
    except json.JSONDecodeError:
        return {"error": "Failed to parse JSON response"}

print("✓ API call function defined")

✓ API call function defined


## Block 4: Define Response Extraction Function

In [4]:
def extract_assistant_reply(response):
    """
    Extract the assistant's text reply from API response
    
    Args:
        response: API response dictionary
    
    Returns:
        str: Extracted text or error message
    """
    try:
        if "error" in response:
            return f"ERROR: {response['error']}"

        if "content" in response and len(response["content"]) > 0:
            return response["content"][0]["text"]

        return "No content in response"
    except Exception as e:
        return f"Failed to extract reply: {str(e)}"

print("✓ Response extraction function defined")

✓ Response extraction function defined


## Block 5: Query 1 - Technical Question (Deterministic)

In [5]:
print("\n" + "=" * 80)
print("QUERY 1: TECHNICAL QUESTION")
print("=" * 80)

query1 = "What is the difference between supervised and unsupervised learning in machine learning?"

print(f"\n[Input Query]:")
print(f"  \"{query1}\"")
print(f"\n[Parameters]:")
print(f"  Temperature: 0.5 (More deterministic - focused answers)")
print(f"  Max Tokens: 512")

response1 = call_llm_api(query1, temperature=0.5, max_tokens=512)

print(f"\n✓ Response received!")


QUERY 1: TECHNICAL QUESTION

[Input Query]:
  "What is the difference between supervised and unsupervised learning in machine learning?"

[Parameters]:
  Temperature: 0.5 (More deterministic - focused answers)
  Max Tokens: 512
  Sending request to API...

✓ Response received!


## Block 6: Display Query 1 Response

In [6]:
print("\n[Full JSON Response]:")
print(json.dumps(response1, indent=2))

reply1 = extract_assistant_reply(response1)
print(f"\n[Assistant Reply]:")
print(f"  {reply1}")

if "usage" in response1:
    print(f"\n[Token Usage]:")
    print(f"  Input tokens: {response1['usage']['input_tokens']}")
    print(f"  Output tokens: {response1['usage']['output_tokens']}")
    print(f"  Total: {response1['usage']['input_tokens'] + response1['usage']['output_tokens']}")


[Full JSON Response]:
{
  "error": "HTTP Error 401: {\"type\":\"error\",\"error\":{\"type\":\"authentication_error\",\"message\":\"invalid x-api-key\"},\"request_id\":\"req_011CcMuZMcxyBri2jkVENV11\"}"
}

[Assistant Reply]:
  ERROR: HTTP Error 401: {"type":"error","error":{"type":"authentication_error","message":"invalid x-api-key"},"request_id":"req_011CcMuZMcxyBri2jkVENV11"}


## Block 7: Query 2 - Creative Question (High Temperature)

In [7]:
print("\n" + "=" * 80)
print("QUERY 2: CREATIVE QUESTION (Higher Temperature)")
print("=" * 80)

query2 = "Suggest creative ideas for an AI-based product that helps developers learn better."

print(f"\n[Input Query]:")
print(f"  \"{query2}\"")
print(f"\n[Parameters]:")
print(f"  Temperature: 0.9 (More creative - exploratory answers)")
print(f"  Max Tokens: 512")

response2 = call_llm_api(query2, temperature=0.9, max_tokens=512)

print(f"\n✓ Response received!")


QUERY 2: CREATIVE QUESTION (Higher Temperature)

[Input Query]:
  "Suggest creative ideas for an AI-based product that helps developers learn better."

[Parameters]:
  Temperature: 0.9 (More creative - exploratory answers)
  Max Tokens: 512
  Sending request to API...

✓ Response received!


## Block 8: Display Query 2 Response

In [8]:
print("\n[Full JSON Response]:")
print(json.dumps(response2, indent=2))

reply2 = extract_assistant_reply(response2)
print(f"\n[Assistant Reply]:")
print(f"  {reply2}")

if "usage" in response2:
    print(f"\n[Token Usage]:")
    print(f"  Input tokens: {response2['usage']['input_tokens']}")
    print(f"  Output tokens: {response2['usage']['output_tokens']}")
    print(f"  Total: {response2['usage']['input_tokens'] + response2['usage']['output_tokens']}")


[Full JSON Response]:
{
  "error": "HTTP Error 401: {\"type\":\"error\",\"error\":{\"type\":\"authentication_error\",\"message\":\"invalid x-api-key\"},\"request_id\":\"req_011CcMuZQnUG2dsEnxqazArm\"}"
}

[Assistant Reply]:
  ERROR: HTTP Error 401: {"type":"error","error":{"type":"authentication_error","message":"invalid x-api-key"},"request_id":"req_011CcMuZQnUG2dsEnxqazArm"}


## Block 9: Query 3 - Dynamic User Input

In [9]:
print("\n" + "=" * 80)
print("QUERY 3: DYNAMIC USER INPUT")
print("=" * 80)

# Example user input (can be modified)
user_input = "What are the top 3 skills every software developer should learn?"

print(f"\n[Example User Query]:")
print(f"  \"{user_input}\"")

# Use custom temperature
temperature = 0.7

print(f"\n[Parameters]:")
print(f"  Temperature: {temperature}")
print(f"  Max Tokens: 512")

response3 = call_llm_api(user_input, temperature=temperature, max_tokens=512)

print(f"\n✓ Response received!")


QUERY 3: DYNAMIC USER INPUT

[Example User Query]:
  "What are the top 3 skills every software developer should learn?"

[Parameters]:
  Temperature: 0.7
  Max Tokens: 512
  Sending request to API...

✓ Response received!


## Block 10: Display Query 3 Response

In [10]:
print("\n[Full JSON Response]:")
print(json.dumps(response3, indent=2))

reply3 = extract_assistant_reply(response3)
print(f"\n[Assistant Reply]:")
print(f"  {reply3}")

if "usage" in response3:
    print(f"\n[Token Usage]:")
    print(f"  Input tokens: {response3['usage']['input_tokens']}")
    print(f"  Output tokens: {response3['usage']['output_tokens']}")
    print(f"  Total: {response3['usage']['input_tokens'] + response3['usage']['output_tokens']}")


[Full JSON Response]:
{
  "error": "HTTP Error 401: {\"type\":\"error\",\"error\":{\"type\":\"authentication_error\",\"message\":\"invalid x-api-key\"},\"request_id\":\"req_011CcMuZUXwVU9N3Zh7i7gZG\"}"
}

[Assistant Reply]:
  ERROR: HTTP Error 401: {"type":"error","error":{"type":"authentication_error","message":"invalid x-api-key"},"request_id":"req_011CcMuZUXwVU9N3Zh7i7gZG"}


## Block 11: Temperature Effect Comparison

In [11]:
print("\n" + "=" * 80)
print("TEMPERATURE EFFECT COMPARISON")
print("=" * 80)

temperature_guide = pd.DataFrame({
    'Temperature Range': ['0.0 - 0.3', '0.4 - 0.6', '0.7 - 1.0'],
    'Behavior': ['Deterministic', 'Balanced', 'Creative'],
    'Use Case': [
        'Technical questions, factual answers',
        'General purpose, balanced responses',
        'Brainstorming, creative tasks'
    ],
    'Example Use': [
        'Math problems, definitions',
        'Regular queries, summaries',
        'Idea generation, storytelling'
    ]
})

print("\n[Temperature Effects]:")
print(temperature_guide.to_string(index=False))

print("\n[Query Comparison]:")
print(f"\n1. Query 1 (Temperature 0.5):")
print(f"   - Type: Technical question")
print(f"   - Response: Structured, consistent, factual")
print(f"   - Tokens used: {response1.get('usage', {}).get('output_tokens', 'N/A')}")

print(f"\n2. Query 2 (Temperature 0.9):")
print(f"   - Type: Creative question")
print(f"   - Response: More varied, exploratory, creative")
print(f"   - Tokens used: {response2.get('usage', {}).get('output_tokens', 'N/A')}")

print(f"\n3. Query 3 (Temperature 0.7):")
print(f"   - Type: General purpose question")
print(f"   - Response: Balanced approach")
print(f"   - Tokens used: {response3.get('usage', {}).get('output_tokens', 'N/A')}")


TEMPERATURE EFFECT COMPARISON

[Temperature Effects]:
Temperature Range      Behavior                             Use Case                   Example Use
        0.0 - 0.3 Deterministic Technical questions, factual answers    Math problems, definitions
        0.4 - 0.6      Balanced  General purpose, balanced responses    Regular queries, summaries
        0.7 - 1.0      Creative        Brainstorming, creative tasks Idea generation, storytelling

[Query Comparison]:

1. Query 1 (Temperature 0.5):
   - Type: Technical question
   - Response: Structured, consistent, factual
   - Tokens used: N/A

2. Query 2 (Temperature 0.9):
   - Type: Creative question
   - Response: More varied, exploratory, creative
   - Tokens used: N/A

3. Query 3 (Temperature 0.7):
   - Type: General purpose question
   - Response: Balanced approach
   - Tokens used: N/A


## Block 12: Error Handling Best Practices

In [12]:
print("\n" + "=" * 80)
print("ERROR HANDLING BEST PRACTICES")
print("=" * 80)

error_handling = """
1. TIMEOUT ERRORS:
   - API request takes too long (> 30 seconds)
   - Solution: Set appropriate timeout and retry logic
   - Catch: requests.exceptions.Timeout

2. HTTP ERRORS:
   - 401: Authentication failed (invalid API key)
   - 429: Rate limit exceeded
   - 500: Server error
   - Solution: Check API key, implement backoff strategy
   - Catch: requests.exceptions.HTTPError

3. NETWORK ERRORS:
   - Connection refused or network unreachable
   - Solution: Check internet connection, implement retry
   - Catch: requests.exceptions.RequestException

4. PARSING ERRORS:
   - Response is not valid JSON
   - Solution: Validate response format
   - Catch: json.JSONDecodeError

5. RESPONSE EXTRACTION:
   - Missing expected fields in response
   - Solution: Use safe extraction with defaults
   - Implementation: Check for 'content' field before access
"""

print(error_handling)


ERROR HANDLING BEST PRACTICES

1. TIMEOUT ERRORS:
   - API request takes too long (> 30 seconds)
   - Solution: Set appropriate timeout and retry logic
   - Catch: requests.exceptions.Timeout

2. HTTP ERRORS:
   - 401: Authentication failed (invalid API key)
   - 429: Rate limit exceeded
   - 500: Server error
   - Solution: Check API key, implement backoff strategy
   - Catch: requests.exceptions.HTTPError

3. NETWORK ERRORS:
   - Connection refused or network unreachable
   - Solution: Check internet connection, implement retry
   - Catch: requests.exceptions.RequestException

4. PARSING ERRORS:
   - Response is not valid JSON
   - Solution: Validate response format
   - Catch: json.JSONDecodeError

5. RESPONSE EXTRACTION:
   - Missing expected fields in response
   - Solution: Use safe extraction with defaults
   - Implementation: Check for 'content' field before access



## Block 13: Key Learnings & Summary

In [13]:
print("\n" + "=" * 80)
print("KEY LEARNINGS & SUMMARY")
print("=" * 80)

learnings = """
1. API-BASED LLM INTERACTION:
   ✓ Send HTTP requests to LLM endpoints
   ✓ Include authentication headers (API key)
   ✓ Structure payload with model, messages, parameters
   ✓ Handle responses and errors gracefully

2. REQUEST STRUCTURE:
   - Headers: API key, API version, content-type
   - Payload: model, max_tokens, temperature, messages
   - Timeout: Set appropriate timeout (30s default)

3. RESPONSE PARSING:
   - Convert JSON response to dictionary
   - Extract content from 'content' array
   - Check for errors in response
   - Access token usage information

4. PARAMETER TUNING:
   - temperature: Controls randomness (0.0-1.0)
   - max_tokens: Limits response length
   - Higher temp = more creative
   - Lower temp = more focused

5. PROMPT ENGINEERING:
   ✓ Clear and specific questions get better answers
   ✓ Provide context when needed
   ✓ Structure complex queries well
   ✓ Use appropriate temperature for task type

6. PRODUCTION CONSIDERATIONS:
   ✓ Implement retry logic for failed requests
   ✓ Rate limiting and backoff strategies
   ✓ Log API usage and costs
   ✓ Monitor token usage
   ✓ Cache responses when appropriate
   ✓ Validate and sanitize user inputs
"""

print(learnings)
print("\n" + "=" * 80)
print("✓ Exercise 2 completed successfully!")
print("=" * 80)


KEY LEARNINGS & SUMMARY

1. API-BASED LLM INTERACTION:
   ✓ Send HTTP requests to LLM endpoints
   ✓ Include authentication headers (API key)
   ✓ Structure payload with model, messages, parameters
   ✓ Handle responses and errors gracefully

2. REQUEST STRUCTURE:
   - Headers: API key, API version, content-type
   - Payload: model, max_tokens, temperature, messages
   - Timeout: Set appropriate timeout (30s default)

3. RESPONSE PARSING:
   - Convert JSON response to dictionary
   - Extract content from 'content' array
   - Check for errors in response
   - Access token usage information

4. PARAMETER TUNING:
   - temperature: Controls randomness (0.0-1.0)
   - max_tokens: Limits response length
   - Higher temp = more creative
   - Lower temp = more focused

5. PROMPT ENGINEERING:
   ✓ Clear and specific questions get better answers
   ✓ Provide context when needed
   ✓ Structure complex queries well
   ✓ Use appropriate temperature for task type

6. PRODUCTION CONSIDERATIONS:
   ✓ 